NextGen Learners Internship Week 1 — Foundations: Cleaning & Exploring the Applicant Dataset by Mwelwa Kashingwa

In [19]:
!pip install rapidfuzz

In [20]:
import pandas as pd
import numpy as np
from rapidfuzz import process

In [21]:
from google.colab import files
uploaded = files.upload()

Saving applicants.csv to applicants (1).csv


In [22]:
df = pd.read_csv('applicants.csv')
df.shape # how many rows and columns?
df.head() # preview the first 5 rows
df.info() # data types + non-null counts per column
df.isnull().sum() # exact count of missing values per column
df.duplicated().sum() # how many exact duplicate rows exist

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85 entries, 0 to 84
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Applicant Name    83 non-null     object
 1   Email             79 non-null     object
 2   Phone             85 non-null     object
 3   Domain Applied    85 non-null     object
 4   University        81 non-null     object
 5   Application Date  85 non-null     object
 6   Status            82 non-null     object
dtypes: object(7)
memory usage: 4.8+ KB


np.int64(3)

The dataset contains 85 rows and 7 columns. The 7 columns are of type object because their entries are strings. Applicant name has 2 missing values, email has 6 missing values, university has 4 missing values, and status 3 missing cells. While phone, domain applied, and application date have no missing values. There are 3 exact duplicate rows.

In [24]:
# Dropping rows with missing applicant name because a nameless application isn't usable for outreach.
df = df.dropna(subset=['Applicant Name'])
df['Applicant Name'] = df['Applicant Name'].str.strip().str.title() # Set a consistent format for the applicant names
df.isnull().sum() # exact count of missing values per column

,0
Applicant Name,0
Email,6
Phone,0
Domain Applied,0
University,4
Application Date,0
Status,3


In [25]:
# Replacing missing emails with like "not_provided@example.com"
df['Email'] = df['Email'].fillna('not_provided@example.com')
df['Email'] = df['Email'].str.strip().str.lower() # Set a consistent format for the emails
df.isnull().sum()

,0
Applicant Name,0
Email,0
Phone,0
Domain Applied,0
University,4
Application Date,0
Status,3


In [26]:
# Replacing missing university with 'Not specified'
# I filled missing University with 'Not Specified' instead of dropping, since it's not essential to determining application status.
df['University'] = df['University'].fillna('Not Specified')
df.isnull().sum()

,0
Applicant Name,0
Email,0
Phone,0
Domain Applied,0
University,0
Application Date,0
Status,3


In [27]:
# Missing status filled with Under review
df['Status'] = df['Status'].fillna('Under Review')
# Changing the status entries to a consistent format
df['Status'] = df['Status'].str.capitalize()
df.isnull().sum()

,0
Applicant Name,0
Email,0
Phone,0
Domain Applied,0
University,0
Application Date,0
Status,0


In [28]:
# Change phone numbers to a consistent format so that it can be used as a key, email would not
# be viable because applicants may have missing emails assigned the same placeholder address.
# If a phone number appears more than once, keep the first entry and remove the rest.
df['Phone'] = df['Phone'].astype('string')
df['Phone'] = df['Phone'].str.replace('-', '', regex=False).str.replace(' ', '', regex=False)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 83 entries, 0 to 84
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Applicant Name    83 non-null     object
 1   Email             83 non-null     object
 2   Phone             83 non-null     string
 3   Domain Applied    83 non-null     object
 4   University        83 non-null     object
 5   Application Date  83 non-null     object
 6   Status            83 non-null     object
dtypes: object(6), string(1)
memory usage: 5.2+ KB


In [29]:
# Remove duplicates with phone number as key
df = df.drop_duplicates(subset=['Phone'], keep='first')
print(df.shape)
df.duplicated().sum()

(78, 7)


np.int64(0)

In [30]:
# Create consistency in the format of domain applied
DOMAIN_MAPPING = {
    'data science': 'Data Science',
    'data sceince': 'Data Science',  # Handles the typo
    'cybersecurity': 'Cybersecurity',
    'cyber security': 'Cybersecurity',
    'web development': 'Web Development',
    'web dev': 'Web Development',
    'webdev': 'Web Development',
    'mobile app development': 'Mobile App Development',
    'mobile app dev': 'Mobile App Development',
    'ui/ux design': 'UI/UX Design',
    'ui ux design': 'UI/UX Design',  # Made lowercase
    'ui-ux design': 'UI/UX Design'
}


def clean_domain(text):
    if pd.isna(text):
        return text


    clean_text = " ".join(str(text).split()).lower()

    # Return the exact category name
    return DOMAIN_MAPPING.get(clean_text, text)


df['Domain Applied'] = df['Domain Applied'].apply(clean_domain)


print("Unique domains left:")
print(df['Domain Applied'].unique())

df.head()

Unique domains left:
['Web Development' 'Data Science' 'Cybersecurity' 'Mobile App Development'
 'UI/UX Design']


,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
0,Farah Siddiqui,farah.siddiqui46@outlook.com,03210471235,Web Development,IBA,"Apr 11, 2026",Under review
1,Usman Ahmed,usman.ahmed@hotmail.com,03122863352,Data Science,University of the Punjab,16 Apr 2026,Under review
2,Areeba Zaidi,areeba.zaidi69@hotmail.com,03467099740,Cybersecurity,IBA Karachi,2026-04-30,Selected
3,Ayesha Zaidi,ayesha.zaidi@hotmail.com,03358083375,Data Science,IBA Karachi,"Apr 06, 2026",Under review
4,Daniyal Javed,daniyal.javed@outlook.com,03452651916,Cybersecurity,IBA Karachi,2026/04/08,Rejected


In [31]:
# Standardize the Status column
df['Status'] = df['Status'].str.strip().str.title()

In [34]:
#Convert Application Date to a real date type
df['Application Date'] = pd.to_datetime( df['Application Date'], format='mixed', errors='coerce',
    dayfirst=False)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 78 entries, 0 to 84
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Applicant Name    78 non-null     object        
 1   Email             78 non-null     object        
 2   Phone             78 non-null     string        
 3   Domain Applied    78 non-null     object        
 4   University        78 non-null     object        
 5   Application Date  76 non-null     datetime64[ns]
 6   Status            78 non-null     object        
dtypes: datetime64[ns](1), object(5), string(1)
memory usage: 4.9+ KB


Data Quality Summary

Starting dataset: 85 rows, 7 columns.

Issues found:
- 2 missing values in Applicant name, 6 in Email, 4 in University, 3 in Status
- 3 exact duplicate rows identified by matching Phone
- Domain Applied had 9 different spelling/casing variations for
only 5 real domains
- Application Date was stored as text, not as an actual date type
- Phone numbers contained inconsistent dash/space formatting

Actions taken:
- Dropped 2 rows with missing Applicant Name (unusable without identification)
- Filled missing University with "Not Specified" (non-critical field)
- Filled missing Status with "Under Review" (default for unprocessed applications)
- Removed 3 duplicate rows total
- Standardized all Domain Applied values into 5 consistent categories
- Converted Application Date to proper datetime format
- Cleaned Phone number formatting for consistency
Final dataset: 78 rows, 7 columns, no missing values in critical
fields, no duplicates, consistent formatting throughout.

In [35]:
df.to_csv('applicants_cleaned.csv', index=False)